In [4]:
# -*- coding: utf-8 -*-
import re
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from warnings import filterwarnings
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score

filterwarnings("ignore")
RANDOM_STATE = 111
np.random.seed(RANDOM_STATE)

def extract_number(x):
    if pd.isna(x): return np.nan
    m = re.search(r"(-?\d+\.?\d*)", str(x))
    return float(m.group(1)) if m else np.nan

def parse_room_layout(s):
    s = str(s)
    rm = re.search(r"(\d+)室", s)
    lr = re.search(r"(\d+)厅", s)
    bt = re.search(r"(\d+)卫", s)
    return (
        float(rm.group(1)) if rm else np.nan,
        float(lr.group(1)) if lr else np.nan,
        float(bt.group(1)) if bt else np.nan,
    )

def parse_floor_ratio(s):
    s = str(s)
    nums = re.findall(r"(\d+\.?\d*)", s)
    if len(nums) >= 2:
        top, den = float(nums[0]), float(nums[-1])
        return top / den if den != 0 else np.nan
    elif len(nums) == 1:
        return float(nums[0])
    return np.nan

def extract_year(s):
    s = str(s)
    m = re.search(r"(19|20)\d{2}", s)
    return int(m.group(0)) if m else np.nan

def iqr_clip(sr):
    q1, q3 = np.nanpercentile(sr, [25, 75])
    iqr = q3 - q1
    return np.clip(sr, q1 - 1.5 * iqr, q3 + 1.5 * iqr)

def clean_train(df, task):
    df = df.copy()
    drop_cols = ["开发商","物业公司","物业办公电话","产权描述","客户反馈","coord_x","coord_y","房屋优势","核心卖点","户型介绍","周边配套","交通出行"]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors="ignore")
    for c in ["面积", "建筑面积", "套内面积"]:
        if c in df.columns: df[c] = df[c].apply(extract_number)
    if "房屋户型" in df.columns:
        df[["室","厅","卫"]] = df["房屋户型"].apply(lambda s: pd.Series(parse_room_layout(s)))
        df.drop(columns=["房屋户型"], inplace=True)
    if "所在楼层" in df.columns:
        df["楼层比例"] = df["所在楼层"].apply(parse_floor_ratio)
    if "建筑年代" in df.columns:
        df["建筑年代"] = df["建筑年代"].apply(extract_year)
        df["房龄"] = 2025 - df["建筑年代"]
    if {"lon", "lat"}.issubset(df.columns):
        df["lon_std"] = (df["lon"] - df["lon"].mean()) / (df["lon"].std() or 1)
        df["lat_std"] = (df["lat"] - df["lat"].mean()) / (df["lat"].std() or 1)
        df["lon_lat_interaction"] = df["lon_std"] * df["lat_std"]
    if {"面积","房龄"}.issubset(df.columns):
        df["面积房龄交互"] = np.log1p(df["面积"]) * df["房龄"]
    if "Price" in df.columns:
        df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
        if df["Price"].median() < 1000: df["Price"] *= 10000
        df["Price"] = iqr_clip(df["Price"])
    cat_cols = [c for c in ["城市","区域","区县","板块","装修","朝向","物业类别","建筑结构","租赁方式","付款方式","电梯","车位","环线位置"] if c in df.columns]
    for c in cat_cols: df[c] = df[c].astype(str).fillna("未知")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df = df.fillna(df.median(numeric_only=True)).fillna(0)
    return df

def train_models(df, task):
    print(f"\n🚀 开始训练 {task.upper()} 模型，共 {df.shape[0]} 条样本")
    y = df["Price"].values
    X = df.drop(columns=["Price"], errors="ignore").select_dtypes(include=[np.number])
    pt_y = PowerTransformer(method='yeo-johnson')
    y_trans = pt_y.fit_transform(y.reshape(-1, 1)).flatten()
    joblib.dump(pt_y, f"target_transformer_{task}.pkl")
    X_train, X_test, y_train, y_test = train_test_split(X, y_trans, test_size=0.2, random_state=RANDOM_STATE)
    joblib.dump(X.columns.tolist(), f"model_features_{task}.pkl")
    core_poly = [c for c in ["面积","房龄","楼层比例","lon_std","lat_std"] if c in X.columns]
    preproc = ColumnTransformer([
        ("poly", PolynomialFeatures(degree=2, include_bias=False), core_poly),
        ("pass", "passthrough", X.columns)
    ], remainder='drop')
    models = {"OLS": LinearRegression(), "Ridge": Ridge(random_state=RANDOM_STATE),
              "LASSO": Lasso(max_iter=20000, random_state=RANDOM_STATE),
              "ElasticNet": ElasticNet(max_iter=20000, random_state=RANDOM_STATE)}
    params = {
        "Ridge": {"model__alpha": np.logspace(-2, 2, 6)},
        "LASSO": {"model__alpha": np.logspace(-3, 1, 5)},
        "ElasticNet": {"model__alpha": np.logspace(-3, 1, 5), "model__l1_ratio": [0.3, 0.5, 0.7]},
    }
    inv = lambda arr: pt_y.inverse_transform(arr.reshape(-1,1)).flatten()
    results = []
    cv = KFold(n_splits=6, shuffle=True, random_state=RANDOM_STATE)
    for name, est in models.items():
        print(f"\n🔹 训练 {name} ...")
        pipe = Pipeline([("prep", preproc), ("scaler", StandardScaler()), ("model", est)])
        if name in params:
            grid = GridSearchCV(pipe, params[name], cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
            grid.fit(X_train, y_train)
            model = grid.best_estimator_
        else:
            model = pipe.fit(X_train, y_train)
        y_pred_in = inv(model.predict(X_train))
        y_pred_out = inv(model.predict(X_test))
        y_true_in = inv(y_train)
        y_true_out = inv(y_test)
        mae_in, mae_out = mean_absolute_error(y_true_in, y_pred_in), mean_absolute_error(y_true_out, y_pred_out)
        r2_in, r2_out = r2_score(y_true_in, y_pred_in), r2_score(y_true_out, y_pred_out)
        cv_mae = -cross_val_score(model, X, y_trans, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1).mean()
        results.append({"Model": name, "In-sample MAE": mae_in, "In-sample R²": r2_in, "Out-of-sample MAE": mae_out, "Out-of-sample R²": r2_out, "CV (6-fold) MAE": cv_mae, "Samples (after outlier removal)": X.shape[0]})
        plt.figure(figsize=(6,6))
        plt.scatter(y_true_out, y_pred_out, s=10, alpha=0.5)
        lo, hi = float(np.min(y_true_out)), float(np.max(y_true_out))
        plt.plot([lo, hi], [lo, hi], 'r--', lw=2)
        plt.xlabel("真实价格"); plt.ylabel("预测价格")
        plt.title(f"{task.upper()} - {name}（测试集）")
        plt.tight_layout(); plt.savefig(f"scatter_{task}_{name}.png", dpi=150); plt.close()
        plt.figure(figsize=(6,4))
        plt.hist(y_pred_out - y_true_out, bins=40, edgecolor="k")
        plt.xlabel("预测误差"); plt.ylabel("频数")
        plt.title(f"{task.upper()} - {name} 误差分布")
        plt.tight_layout(); plt.savefig(f"hist_{task}_{name}.png", dpi=150); plt.close()
        joblib.dump(model, f"model_{task}_{name}.pkl")
    pd.DataFrame(results).to_csv(f"performance_summary_{task}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ 已保存 performance_summary_{task}.csv")

if __name__ == "__main__":
    price_df = pd.read_excel("ruc_Class25Q2_train_price.xlsx")
    rent_df = pd.read_excel("ruc_Class25Q2_train_rent.xlsx")
    df_price = clean_train(price_df, "price")
    df_rent = clean_train(rent_df, "rent")
    train_models(df_price, "price")
    train_models(df_rent, "rent")



🚀 开始训练 PRICE 模型，共 103871 条样本

🔹 训练 OLS ...

🔹 训练 Ridge ...

🔹 训练 LASSO ...

🔹 训练 ElasticNet ...
✅ 已保存 performance_summary_price.csv

🚀 开始训练 RENT 模型，共 98899 条样本

🔹 训练 OLS ...

🔹 训练 Ridge ...

🔹 训练 LASSO ...

🔹 训练 ElasticNet ...
✅ 已保存 performance_summary_rent.csv


In [5]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import joblib
import re

def extract_number(x):
    if pd.isna(x): return np.nan
    m = re.search(r"(-?\d+\.?\d*)", str(x))
    return float(m.group(1)) if m else np.nan

def parse_floor_ratio(s):
    s = str(s)
    nums = re.findall(r"(\d+\.?\d*)", s)
    if len(nums) >= 2:
        top, den = float(nums[0]), float(nums[-1])
        return top / den if den != 0 else np.nan
    elif len(nums) == 1:
        return float(nums[0])
    return np.nan

def extract_year(s):
    s = str(s)
    m = re.search(r"(19|20)\d{2}", s)
    return int(m.group(0)) if m else np.nan

def clean_test(df_raw, task):
    df = df_raw.copy()
    if "ID" not in df.columns:
        raise KeyError("❌ 测试集缺少 'ID' 列")
    for c in ["面积", "建筑面积", "套内面积"]:
        if c in df.columns: df[c] = df[c].apply(extract_number)
    if "所在楼层" in df.columns:
        df["楼层比例"] = df["所在楼层"].apply(parse_floor_ratio)
    if "建筑年代" in df.columns:
        df["建筑年代"] = df["建筑年代"].apply(extract_year)
        df["房龄"] = 2025 - df["建筑年代"]
    if {"lon", "lat"}.issubset(df.columns):
        lon_std = (df["lon"] - df["lon"].mean()) / (df["lon"].std() or 1)
        lat_std = (df["lat"] - df["lat"].mean()) / (df["lat"].std() or 1)
        df["lon_std"], df["lat_std"] = lon_std, lat_std
        df["lon_lat_interaction"] = lon_std * lat_std
    if {"面积", "房龄"}.issubset(df.columns):
        df["面积房龄交互"] = np.log1p(df["面积"]) * df["房龄"]
    cat_cols = [c for c in ["城市","区域","区县","板块","装修","朝向","物业类别","建筑结构","租赁方式","付款方式","电梯","车位","环线位置"] if c in df.columns]
    for c in cat_cols: df[c] = df[c].astype(str).fillna("未知")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df = df.fillna(df.median(numeric_only=True)).fillna(0)
    return df

def stabilize_predictions(pred):
    pred = np.clip(pred, 0, None)
    if len(pred) == 0: return pred
    q1, q3 = np.percentile(pred, [10, 90])
    iqr = q3 - q1
    return np.clip(pred, q1 - 0.5 * iqr, q3 + 1.5 * iqr)

def predict_and_merge(model_name, df_price_test, df_rent_test):
    feat_price = joblib.load("model_features_price.pkl")
    feat_rent = joblib.load("model_features_rent.pkl")
    pt_price = joblib.load("target_transformer_price.pkl")
    pt_rent = joblib.load("target_transformer_rent.pkl")
    mdl_price = joblib.load(f"model_price_{model_name}.pkl")
    mdl_rent = joblib.load(f"model_rent_{model_name}.pkl")
    Xp = df_price_test.reindex(columns=feat_price, fill_value=0)
    Xr = df_rent_test.reindex(columns=feat_rent, fill_value=0)
    yp = pt_price.inverse_transform(mdl_price.predict(Xp).reshape(-1,1)).flatten()
    yr = pt_rent.inverse_transform(mdl_rent.predict(Xr).reshape(-1,1)).flatten()
    yp = stabilize_predictions(yp)
    yr = stabilize_predictions(yr)
    sub_p = pd.DataFrame({"ID": df_price_test["ID"], "Price": yp})
    sub_r = pd.DataFrame({"ID": df_rent_test["ID"], "Price": yr})
    sub_all = pd.concat([sub_p, sub_r], axis=0, ignore_index=True).sort_values("ID")
    file_name = f"submission_{model_name}.csv"
    sub_all.to_csv(file_name, index=False, encoding="utf-8-sig")
    print(f"✅ {file_name} 生成，样本 {len(sub_all)} 均价 {np.mean(sub_all.Price):.0f}")

if __name__ == "__main__":
    df_price_raw = pd.read_excel("ruc_Class25Q2_test_price.xlsx")
    df_rent_raw = pd.read_excel("ruc_Class25Q2_test_rent.xlsx")
    df_price_test = clean_test(df_price_raw, "price")
    df_rent_test = clean_test(df_rent_raw, "rent")
    for model in ["OLS", "Ridge", "LASSO", "ElasticNet"]:
        predict_and_merge(model, df_price_test, df_rent_test)


✅ submission_OLS.csv 生成，样本 43790 均价 1344731
✅ submission_Ridge.csv 生成，样本 43790 均价 1340692
✅ submission_LASSO.csv 生成，样本 43790 均价 1326378
✅ submission_ElasticNet.csv 生成，样本 43790 均价 1342007
